In [80]:
import polars as pl
import numpy as np
import plotly.express as px
import joblib
from sklearn.metrics import root_mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Constants
ID = "id"
PREDICTION_FILES = {
    "lasso": "../oof/oof_preds_lasso.csv",
    "rf": "../oof/oof_preds_rf.csv",
    "lgbm": "../oof/oof_preds_lgbm.csv",
    "lasso_log": "../oof/oof_preds_lasso_log.csv",
    "rf_log": "../oof/oof_preds_rf_log.csv",
    "lgbm_log": "../oof/oof_preds_lgbm_log.csv"
}
TARGET_FILE = "../data/salary.csv"
TARGET_COL = "Salary"
RENAMED_TARGET = "y_true"
SEED = 100622

In [82]:
def metrics(y_true, y_pred):
    rmse = root_mean_squared_error(y_true, y_pred) * -1
    r2 = r2_score(y_true, y_pred)
    return {'rmse': rmse, 'r2': r2}

In [83]:
def bootstrap_confidence_intervals(y_true, y_pred, n_bootstrap=1000, confidence_level=0.95):
    """
    Calculate bootstrap confidence intervals for RMSE and R2.

    Args:
        y_true: True values
        y_pred: Predicted values
        n_bootstrap: Number of bootstrap samples
        confidence_level: Confidence level

    Returns:
        Dictionary with bootstrap confidence intervals
    """
    n_samples = len(y_true)
    bootstrap_rmse = []
    bootstrap_r2 = []

    np.random.seed(SEED)
    for _ in range(n_bootstrap):
        # Bootstrap sample indices
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        # Calculate metrics for bootstrap sample
        rmse_boot = root_mean_squared_error(y_true_boot, y_pred_boot)
        r2_boot = r2_score(y_true_boot, y_pred_boot)

        bootstrap_rmse.append(rmse_boot)
        bootstrap_r2.append(r2_boot)

    # Calculate percentile confidence intervals
    alpha = 1 - confidence_level
    lower_percentile = (alpha/2) * 100
    upper_percentile = (1 - alpha/2) * 100

    rmse_ci = np.percentile(bootstrap_rmse, [lower_percentile, upper_percentile])
    r2_ci = np.percentile(bootstrap_r2, [lower_percentile, upper_percentile])

    return {
        'rmse': {
            'mean': np.mean(bootstrap_rmse),
            'std': np.std(bootstrap_rmse),
            'lower_ci': rmse_ci[0],
            'upper_ci': rmse_ci[1]
        },
        'r2': {
            'mean': np.mean(bootstrap_r2),
            'std': np.std(bootstrap_r2),
            'lower_ci': r2_ci[0],
            'upper_ci': r2_ci[1]
        }
    }

In [84]:
def print_metrics_with_ci(results):
  print("\nBootstrap Confidence Intervals for Out-of-Fold Predictions:")
  print("=" * 60)
  print(f"RMSE: {results['rmse']['mean']:.2f}")
  print(f"      95% CI: [{results['rmse']['lower_ci']:.2f}, {results['rmse']['upper_ci']:.2f}]")
  print(f"R²:   {results['r2']['mean']:.3f}")
  print(f"      95% CI: [{results['r2']['lower_ci']:.3f}, {results['r2']['upper_ci']:.3f}]")

In [85]:
def plot_error_vs_predicted(oof_preds_rf, y, ids):
    errors = pl.DataFrame({
        'id': ids,
        'actual': y,
        'predicted': oof_preds_rf
    })
    # calculate the mean and std of the errors
    errors = errors.with_columns(
        ((pl.col('predicted') - pl.col('actual')) ** 2).alias('squared_error')
    )

    rmse_value = np.sqrt(errors['squared_error'].mean())

    fig = px.scatter(
        errors.to_pandas(),
        x='predicted',
        y='actual',
        hover_data=['id'],
        labels={'predicted': 'Predicted Values', 'actual': 'Real Values'},
        title=f'Real vs Predicted Values\nMean Error: {rmse_value:.2f}'
    )
    # add a line at 45 degrees

    max_val = max(errors['predicted'].max(), errors['actual'].max())

    fig.add_shape(
        type='line',
        x0=0,
        y0=0,
        x1=max_val,
        y1=max_val,
        line=dict(
            color='Red',
            width=2,
            dash='dash',
        )
    )
    fig.update_layout(
        width=800,
        height=800,
        legend_title_text='',
        showlegend=False
    )
    fig.show()

In [86]:
def load_prediction(file_path: str, model_name: str) -> pl.DataFrame:
    """Reads a prediction file and renames the predicted column."""
    return pl.read_csv(file_path).rename({"predicted": f"predicted_{model_name}"})

# Load all predictions into a list of DataFrames
prediction_dfs = [load_prediction(path, name) for name, path in PREDICTION_FILES.items()]

# Join all predictions on the ID column
def join_dataframes_on_id(dataframes: list[pl.DataFrame], id_col: str = ID) -> pl.DataFrame:
    """Joins a list of DataFrames on the specified ID column."""
    base = dataframes[0]
    for df in dataframes[1:]:
        base = base.join(df, on=id_col, how="inner")
    return base

# Load target and rename column
y_true = pl.read_csv(TARGET_FILE).rename({TARGET_COL: RENAMED_TARGET})

# Merge all together
oof = join_dataframes_on_id(prediction_dfs + [y_true])

oof

id,predicted_lasso,predicted_rf,predicted_lgbm,y_true
i64,f64,f64,f64,f64
0,62078.420261,65029.225434,62187.087657,90000.0
1,59750.511905,61128.286684,59598.224593,65000.0
2,160369.63358,149486.311217,155823.208796,150000.0
3,71236.655468,61291.847501,56458.42738,60000.0
4,182515.609706,174230.275778,191080.162536,200000.0
…,…,…,…,…
370,84664.159627,85281.568125,83056.357217,85000.0
371,158842.195519,168653.950322,168559.177861,170000.0
372,44321.305468,40217.496668,40937.67544,40000.0


In [87]:
# oof_preds_lasso = pl.read_csv('../oof/oof_preds_lasso.csv').rename({'predicted': 'predicted_lasso'})
# oof_preds_rf = pl.read_csv('../oof/oof_preds_rf.csv').rename({'predicted': 'predicted_rf'})
# oof_preds_lgbm = pl.read_csv('../oof/oof_preds_lgbm.csv').rename({'predicted': 'predicted_lgbm'})
# oof_preds_lasso_log = pl.read_csv('../oof/oof_preds_lasso_log.csv').rename({'predicted': 'predicted_lasso_log'})
# oof_preds_rf_log = pl.read_csv('../oof/oof_preds_rf_log.csv').rename({'predicted': 'predicted_rf_log'})
# oof_preds_lgbm_log = pl.read_csv('../oof/oof_preds_lgbm_log.csv').rename({'predicted': 'predicted_lgbm_log'})

# y_true = pl.read_csv('../data/salary.csv').rename({'Salary': 'y_true'})

# # Join all predictions on 'id'
# oof = oof_preds_lasso.join(
#     oof_preds_rf,
#     on=ID,
#     how='inner'
# ).join(
#     oof_preds_lgbm,
#     on=ID,
#     how='inner'
# ).join(
#     y_true,
#     on=ID,
#     how='inner'
# )

# oof

In [88]:
def ensemble_weights(oof, step=0.01, threshold=1):
    
    model_names = [col for col in oof.columns if 'predicted' in col]

    oof_preds = oof.drop(ID, 'y_true').to_numpy()
    y_true = oof['y_true'].to_numpy()
    scores = []
    
    for col in range(oof_preds.shape[1]):
        score = metrics(y_true, oof_preds[:, col])['rmse']
        scores.append(score)

    sorted_indexes = np.argsort(scores)[::-1]
    sorted_models = oof_preds[:, sorted_indexes]
    sorted_model_names = [model_names[i] for i in sorted_indexes]

    current_best_ensemble = oof_preds[:, sorted_indexes[0]]
    models = sorted_models[:,1:]
    
    history = [metrics(y_true, current_best_ensemble)['rmse']]

    weight_range = np.arange(-0, 0.5, step)

    print(f'Initial model: {sorted_model_names[0]}')
    print(f'RMSE: {history[0]}')

    sorted_model_names.pop(0)
    
    while True:
        best_score = history[-1]
        best_weight = None
        best_model = None
        print(50*'-')
        print(f'Current ensemble: {history[-1]}')
        

        for i in range(models.shape[1]):

            for weight in weight_range:
                ensemble = (1 - weight) * current_best_ensemble + weight * models[:, i]
                score = metrics(y_true, ensemble)['rmse']

                if score > best_score:
                    best_score = score
                    best_weight = weight
                    best_model = i

        if best_model is not None:
            improvement = best_score - history[-1]
            # Calculate percentage improvement relative to current best score
            percentage_improvement = (improvement / abs(history[-1])) * 100
            
            if percentage_improvement > threshold:
                current_best_ensemble = (1 - best_weight) * current_best_ensemble + best_weight * models[:, best_model]
                

                print(f'Best ensemble: {best_score} by adding {sorted_model_names[best_model]} with weight {best_weight}')
                print(f'Improvement: {improvement:.6f} ({percentage_improvement:.3f}%) (threshold: {threshold}%)')
                

                models = np.delete(models, best_model, axis=1)
                sorted_model_names.pop(best_model)
            
                history.append(best_score)
            else:
                print(f'Model {sorted_model_names[best_model]} improvement ({percentage_improvement:.3f}%) below threshold ({threshold}%). Stopping.')
                break
        else:
            print('No improvement found. Stopping.')
            break
    
    return current_best_ensemble

final_ensemble = ensemble_weights(oof)


Initial model: predicted_rf
RMSE: -11425.059633035109
--------------------------------------------------
Current ensemble: -11425.059633035109
Best ensemble: -10855.773585951358 by adding predicted_lasso with weight 0.46
Improvement: 569.286047 (4.983%) (threshold: 1%)
--------------------------------------------------
Current ensemble: -10855.773585951358
Best ensemble: -10700.532512999498 by adding predicted_lgbm with weight 0.27
Improvement: 155.241073 (1.430%) (threshold: 1%)
--------------------------------------------------
Current ensemble: -10700.532512999498
No improvement found. Stopping.
